In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.master("local[*]").appName("DataWarehouseVendas").getOrCreate()

df_vendedores = spark.read.option("header", True).csv("data/Vendedores.csv")
df_transportadoras = spark.read.option("header", True).csv("data/Transportadoras.csv")
df_fornecedores = spark.read.option("header", True).csv("data/Fornecedores.csv")
df_vendas = spark.read.option("header", True).csv("data/Vendas Globais.csv")

from pyspark.sql.types import DoubleType, IntegerType, DateType

df_vendas = df_vendas.withColumn("Vendas Custo", col("Vendas Custo").cast(DoubleType())) \
                     .withColumn("Margem Bruta", col("Margem Bruta").cast(DoubleType())) \
                     .withColumn("Vendas", col("Vendas").cast(DoubleType())) \
                     .withColumn("Desconto", col("Desconto").cast(DoubleType())) \
                     .withColumn("Frete", col("Frete").cast(DoubleType())) \
                     .withColumn("Qtde", col("Qtde").cast(IntegerType())) \
                     .withColumn("Data", col("Data").cast(DateType())) \
                     .withColumn("PedidoID", col("PedidoID").cast(IntegerType()))

df_vendas = df_vendas.withColumn("VendedorID", col("VendedorID").cast(IntegerType())) \
                     .withColumn("TransportadoraID", col("TransportadoraID").cast(IntegerType())) \
                     .withColumn("FornecedorID", col("FornecedorID").cast(IntegerType()))

dim_vendedores = df_vendedores.dropDuplicates(["VendedorID"])
dim_transportadoras = df_transportadoras.dropDuplicates(["TransportadoraID"])
dim_fornecedores = df_fornecedores.dropDuplicates(["FornecedorID"])

dim_clientes = df_vendas.select("ClienteID", "ClienteNome", "ClienteContato", "ClienteCidade", "ClientePaísID", "ClientePaís").dropDuplicates(["ClienteID"])
dim_produtos = df_vendas.select("ProdutoID", "ProdutoNome", "CategoriaID", "CategoriaNome", "CategoriaDescrição").dropDuplicates(["ProdutoID"])
dim_categorias = df_vendas.select("CategoriaID", "CategoriaNome", "CategoriaDescrição").dropDuplicates(["CategoriaID"])

fato_vendas = df_vendas.select(
    "PedidoID", "Data", "VendedorID", "TransportadoraID", "FornecedorID", "ClienteID", "ProdutoID",
    "Vendas Custo", "Margem Bruta", "Vendas", "Desconto", "Frete", "Qtde"
)

In [ ]:
#1. Quem são os meus 10 maiores clientes, em termos de vendas ($)?
top_ten = fato_vendas.join(dim_clientes, "ClienteID") \
    .groupBy("ClienteNome") \
    .sum("Vendas") \
    .orderBy("sum(Vendas)", ascending=False) \
    .limit(10) \
    .toPandas()

#2. Quais os três maiores países, em termos de vendas ($)?

#3. Quais as categorias de produtos que geram maior faturamento (vendas $) no Brasil?

#4. Qual a despesa com frete envolvendo cada transportadora?

#5. Quais são os principais clientes (vendas $) do segmento “Calçados Masculinos”
#(Men ́s Footwear) na Alemanha?

#6. Quais os vendedores que mais dão descontos nos Estados Unidos?

#7. Quais os fornecedores que dão a maior margem de lucro ($) no segmento de
#“Vestuário Feminino” (Womens wear)?

#8. Quanto que foi vendido ($) no ano de 2009? Analisando as vendas anuais entre 2009
#e 2012, podemos concluir que o faturamento vem crescendo, se mantendo estável ou
#decaindo?

#9. Quais são os principais clientes (vendas $) do segmento “Calçados Masculinos”
#(Men ́s Footwear) na Alemanha?

#10. Quais os países nos quais mais se tiram pedidos (qtde total de pedidos)?